# 스텝 단위 데이터로 모델 다시 돌리기 (초안)

| 항목 | 내용 |
|---|---|
| 데이터 | `data/processed/step/` (레시피 스텝 구간 하나 = 한 행, 장비 20대, 244만 행) |
| 목적 | 스텝이 섞이지 않은 데이터 + 스텝별 정상값 기준 z점수로 바꾸면 1분 요약 결과(AUC 0.51~0.57)보다 나아지는지 확인 |
| 상태 | 초안 (튜닝 없음) |

## 1. 방법

- **라벨**: 스텝 구간의 TTF(최소) 3개 중 하나라도 86,400초(24시간) 이하이면 고장 전(1).
- **입력 (누적 카운터 3개는 제외, 센서 13개 × 평균·최소·최대 = 39열)**
  - A. 원래 값
  - B. z점수: (레시피, 스텝)별로 **학습 장비의 정상 구간**에서 구한 평균·표준편차로 표준화. 검증 장비 정보는 기준에 쓰지 않는다.
  - C. B + 장비별 최근 30개 스텝 구간의 z점수 평균(13열) = 52열
- **모델**: 랜덤포레스트(A·B·C), 아이솔레이션 포레스트(B, 정상만 학습).
- **검증**: 1분 요약 노트북과 같은 장비 4묶음. 학습에 쓰지 않은 장비 전체 행으로 AUC.

In [1]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "models" else Path.cwd()
TTF = [
    "TTF_FlowCool Pressure Dropped Below Limit",
    "TTF_Flowcool Pressure Too High Check Flowcool Pump",
    "TTF_Flowcool leak",
]
SENSORS = [
    "IONGAUGEPRESSURE", "ETCHBEAMVOLTAGE", "ETCHBEAMCURRENT",
    "ETCHSUPPRESSORVOLTAGE", "ETCHSUPPRESSORCURRENT",
    "FLOWCOOLFLOWRATE", "FLOWCOOLPRESSURE",
    "ETCHGASCHANNEL1READBACK", "ETCHPBNGASREADBACK",
    "FIXTURETILTANGLE", "ROTATIONSPEED", "ACTUALROTATIONANGLE",
    "ACTUALSTEPDURATION",
]
RAW = [f"{s}_{st}" for s in SENSORS for st in ["mean", "min", "max"]]
CTX = ["recipe", "recipe_step"]
ROLL = 30
TRAIN_SAMPLE = 300_000
SEED = 0

def show(fig):
    fig.update_layout(template="plotly_white", font=dict(family="Malgun Gothic, sans-serif"))
    fig.show()

## 2. 데이터 불러오기

In [2]:
frames = []
for f in sorted(glob.glob(str(ROOT / "data/processed/step/*.parquet"))):
    d = pd.read_parquet(f, columns=["Tool", "time_start", *CTX, *TTF, *RAW])
    d["y"] = (d[TTF] <= 86400).any(axis=1).astype("int8")
    frames.append(d.drop(columns=TTF))
df = pd.concat(frames, ignore_index=True)
df["recipe_step"] = df["recipe_step"].fillna(-1)
tools = sorted(df["Tool"].unique())

pd.DataFrame({
    "스텝 구간 수": [f"{len(df):,}"],
    "고장 전 비율": [f"{df['y'].mean():.1%}"],
    "(레시피, 스텝) 조합": [df.groupby(CTX).ngroups],
}, index=["step/"])

,스텝 구간 수,고장 전 비율,"(레시피, 스텝) 조합"
step/,"2,437,615",7.4%,5742


## 3. 장비 4묶음

1분 요약 노트북과 같은 묶음이 되도록, 원래 이름(`01M01`) 순서로 섞은 뒤 새 이름(`M01_01`)으로 바꾼다.

In [3]:
orig = {t: f"{t[4:]}{t[:3]}" for t in tools}          # M01_01 -> 01M01
by_orig = sorted(tools, key=lambda t: orig[t])
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(by_orig))
folds = [[by_orig[i] for i in perm[k::4]] for k in range(4)]
pd.DataFrame({"검증 장비": [", ".join(sorted(f)) for f in folds]}, index=[f"묶음 {k+1}" for k in range(4)])

,검증 장비
묶음 1,"M01_03, M01_06, M01_08, M02_04, M02_07"
묶음 2,"M01_05, M01_09, M02_03, M02_05, M02_10"
묶음 3,"M01_01, M01_04, M01_10, M02_01, M02_02"
묶음 4,"M01_02, M01_07, M02_06, M02_08, M02_09"


## 4. 입력 만들기와 학습

In [4]:
def zscore(train_mask):
    """학습 장비 정상 구간에서 (레시피, 스텝)별 기준을 구해 전체를 표준화."""
    ref = df.loc[train_mask & (df["y"] == 0)]
    g = ref.groupby(CTX)[RAW]
    mu, sd = g.mean(), g.std()
    sd = sd.where(sd > 1e-6)
    key = pd.MultiIndex.from_frame(df[CTX])
    z = (df[RAW].to_numpy() - mu.reindex(key).to_numpy()) / sd.reindex(key).to_numpy()
    z = pd.DataFrame(z, columns=[c + "_z" for c in RAW], index=df.index).clip(-10, 10).fillna(0)
    roll = (z[[f"{s}_mean_z" for s in SENSORS]].groupby(df["Tool"])
            .transform(lambda x: x.rolling(ROLL, min_periods=5).mean()).fillna(0))
    roll.columns = [c + "_r30" for c in roll.columns]
    return pd.concat([z, roll], axis=1).astype("float32")

ZCOLS = [c + "_z" for c in RAW]
RCOLS = [f"{s}_mean_z_r30" for s in SENSORS]
INPUTS = {"A 원래 값 (39열)": RAW, "B z점수 (39열)": ZCOLS, "C z점수 + 최근 30스텝 (52열)": ZCOLS + RCOLS}

def make_rf(k):
    return RandomForestClassifier(
        n_estimators=100, max_depth=14, min_samples_leaf=100, max_features="sqrt",
        class_weight="balanced_subsample", n_jobs=-1, random_state=k)

def per_tool_auc(test, score, store):
    for t in test["Tool"].unique():
        m = (test["Tool"] == t).to_numpy()
        if test.loc[m, "y"].nunique() == 2:
            store[t] = roc_auc_score(test.loc[m, "y"], score[m])

res = {name: {"auc": [], "tool": {}, "imp": []} for name in [*INPUTS, "IF z점수 (39열)"]}
perm_imp = None
for k, test_tools in enumerate(folds):
    is_test = df["Tool"].isin(test_tools).to_numpy()
    X = pd.concat([df[["Tool", "y", *RAW]], zscore(~is_test)], axis=1)
    train = X[~is_test].sample(TRAIN_SAMPLE, random_state=k)
    test = X[is_test]
    for name, cols in INPUTS.items():
        model = make_rf(k).fit(train[cols].fillna(0), train["y"])
        score = model.predict_proba(test[cols].fillna(0))[:, 1]
        res[name]["auc"].append(roc_auc_score(test["y"], score))
        per_tool_auc(test, score, res[name]["tool"])
        res[name]["imp"].append(pd.Series(model.feature_importances_, index=cols))
        if k == 0 and name.startswith("B"):
            sub = test.sample(50_000, random_state=0)
            r = permutation_importance(model, sub[cols].fillna(0), sub["y"], scoring="roc_auc",
                                       n_repeats=3, random_state=0, n_jobs=1)
            perm_imp = pd.Series(r.importances_mean, index=cols)
    normal = train[train["y"] == 0]
    iso = IsolationForest(n_estimators=200, max_samples=256, random_state=k, n_jobs=-1).fit(normal[ZCOLS])
    score = -iso.score_samples(test[ZCOLS])
    res["IF z점수 (39열)"]["auc"].append(roc_auc_score(test["y"], score))
    per_tool_auc(test, score, res["IF z점수 (39열)"]["tool"])
    print(f"묶음 {k+1} 완료")

묶음 1 완료


묶음 2 완료


묶음 3 완료


묶음 4 완료


### 4-1. 묶음별 AUC

In [5]:
rows = []
for name, r in res.items():
    s = pd.Series(r["tool"])
    rows.append({"입력": name, **{f"묶음 {k+1}": round(a, 3) for k, a in enumerate(r["auc"])},
                 "평균": round(float(np.mean(r["auc"])), 3), "장비별 중앙값": round(float(s.median()), 3),
                 "AUC>0.5 장비": f"{int((s > .5).sum())}/{len(s)}"})
for name, a, med, cnt in [("(참고) 1분 요약 RF 최고 (C)", [0.415, 0.681, 0.512, 0.443], 0.553, "17/20"),
                          ("(참고) 1분 요약 IF", [0.593, 0.522, 0.614, 0.559], 0.535, "18/20")]:
    rows.append({"입력": name, **{f"묶음 {k+1}": v for k, v in enumerate(a)},
                 "평균": round(float(np.mean(a)), 3), "장비별 중앙값": med, "AUC>0.5 장비": cnt})
summary = pd.DataFrame(rows).set_index("입력")
summary

,묶음 1,묶음 2,묶음 3,묶음 4,평균,장비별 중앙값,AUC>0.5 장비
입력,,,,,,,
A 원래 값 (39열),0.346,0.664,0.488,0.381,0.470,0.507,12/20
B z점수 (39열),0.655,0.595,0.591,0.371,0.553,0.504,13/20
C z점수 + 최근 30스텝 (52열),0.675,0.565,0.554,0.397,0.548,0.512,13/20
IF z점수 (39열),0.364,0.509,0.514,0.489,0.469,0.511,12/20
(참고) 1분 요약 RF 최고 (C),0.415,0.681,0.512,0.443,0.513,0.553,17/20
(참고) 1분 요약 IF,0.593,0.522,0.614,0.559,0.572,0.535,18/20


### 4-2. 장비별 AUC

In [6]:
names = list(res)
order = pd.Series(res[names[1]]["tool"]).sort_values().index
fig = go.Figure()
for name in names:
    s = pd.Series(res[name]["tool"])
    fig.add_bar(x=list(order), y=[s.get(t, np.nan) for t in order], name=name)
fig.add_hline(y=0.5, line_dash="dash", annotation_text="무작위(0.5)")
fig.update_layout(title="장비별 AUC (학습에 쓰지 않은 장비)", xaxis_title="장비", yaxis_title="AUC", barmode="group")
show(fig)

### 4-3. 센서별 중요도 (입력 B)

지니 중요도는 4묶음 평균, 순열 중요도는 묶음 1 검증 장비에서 계산했다. 센서별로 평균·최소·최대를 합쳤다.

In [7]:
def by_sensor(s):
    out = {}
    for name, v in s.items():
        key = next(x for x in SENSORS if name.startswith(x + "_"))
        out[key] = out.get(key, 0) + v
    return pd.Series(out)

gini = by_sensor(pd.concat(res["B z점수 (39열)"]["imp"], axis=1).mean(axis=1))
tab = pd.DataFrame({"지니 B": gini, "순열 B(AUC 감소)": by_sensor(perm_imp)}).sort_values("지니 B", ascending=False)
tab.round(4)

,지니 B,순열 B(AUC 감소)
FLOWCOOLPRESSURE,0.1788,-0.0124
ETCHBEAMVOLTAGE,0.1421,0.0336
ROTATIONSPEED,0.1355,-0.0038
ACTUALROTATIONANGLE,0.0993,0.0172
ETCHSUPPRESSORVOLTAGE,0.0782,-0.0007
ETCHGASCHANNEL1READBACK,0.0651,0.0032
FLOWCOOLFLOWRATE,0.0584,-0.0061
ETCHBEAMCURRENT,0.0558,0.0047
ETCHSUPPRESSORCURRENT,0.0512,0.0007
IONGAUGEPRESSURE,0.0475,0.0022


In [8]:
t = tab.iloc[::-1]
fig = go.Figure()
fig.add_bar(y=t.index, x=t["지니 B"], orientation="h", name="지니 중요도")
fig.add_bar(y=t.index, x=t["순열 B(AUC 감소)"], orientation="h", name="순열 중요도 (AUC 감소)")
fig.update_layout(title="센서별 중요도 (입력 B)", barmode="group", height=600, xaxis_title="중요도")
show(fig)

## 5. 해석과 한계

- `recipe_step` 기준 step 데이터로 돌려도 장비 단위 검증 AUC는 0.47~0.55로 무작위(0.5) 근처다. **스텝 섞임을 고쳐도 고장 24시간 전을 구분하는 신호는 거의 보이지 않는다.**
- z점수(스텝별 정상값 기준)로 바꾸면 랜덤포레스트가 0.470 → 0.553으로 오른다. 스텝 차이를 걷어내는 것 자체는 효과가 있다. 그래도 묶음별 편차가 크고(0.37~0.66), 장비 20대 중 13대만 0.5를 넘는다.
- 최근 30스텝 평균(C)은 도움이 되지 않았다(0.548).
- 아이솔레이션 포레스트는 z점수 입력에서 오히려 낮아졌다(1분 요약 0.572 → 0.469).
- 중요도: 지니 상위는 냉각 압력·빔 전압·회전 속도인데, 순열 중요도는 대부분 0 근처이거나 음수다. 검증 장비에서 일관되게 쓸모 있는 센서는 아직 없다.
- 한계: 라벨 창(24시간)이 너무 넓을 수 있다. 고장 시각은 정비로 멈춘 시각이라 실제 이상 시작과 다를 수 있다. 식각 중이 아닌 구간(셔터 위치 ≠ 1)도 섞여 있다. 장비가 20대뿐이라 묶음 결과가 크게 흔들린다.
- 다음 후보: 셔터 위치 1(식각 중)만 사용, 라벨 창을 1~6시간으로 줄여 비교, 고장 종류별(특히 압력 저하) 모델, 고장 직전 몇 시간의 추세(기울기) 피처.